In [31]:
import pandas as pd
import math

In [8]:
MINORITIES = ['Black', 'Latino', 'Other']

In [21]:
def calc_cvap_proportion(cvap_dict, state='AR'):
    porportion = {}
    for minority in MINORITIES:
        porportion[minority] = {}
    
    for district in cvap_dict:
        total = cvap_dict[district]['Total CVAP'] if state=='AR' else cvap_dict[district]['Total Population'] 
        for minority in MINORITIES:
            if state=='AR':
                pop = cvap_dict[district][f'{minority} CVAP']
            else:
                pop = cvap_dict[district][minority]
            porportion[minority][int(district)] = pop/total
        
    return porportion

In [36]:
def calc_cvap_state_proportion(cvap_dict, state='AR'):
    total = 0
    minority_pop = {}
    
    for minority in MINORITIES:
        minority_pop[minority] = 0
        
    for district in cvap_dict:
        total += cvap_dict[district]['Total CVAP'] if state=='AR' else cvap_dict[district]['Total Population'] 
        for minority in MINORITIES:
            if state=='AR':
                minority_pop[minority] += cvap_dict[district][f'{minority} CVAP']
            else:
                minority_pop[minority] += cvap_dict[district][minority]
    
    overall_pro = {}
    for minority in MINORITIES:
        overall_pro[minority] = minority_pop[minority]/total
    return overall_pro

## Arkansas

In [2]:
ar_cvap = pd.read_csv('output/Arkansas/ar-district-CVAP.csv')
ar_cvap

,Unnamed: 0,State,District,Total CVAP,White CVAP,Black CVAP,Latino CVAP,Other CVAP
0,0,Arkansas,1,568995,446068,90719,13902,18306
1,1,Arkansas,2,569348,414225,114846,19625,20652
2,2,Arkansas,3,543233,436910,14522,52980,38821
3,3,Arkansas,4,564359,411979,110143,23680,18557


In [67]:
black_total_ar = ar_cvap['Black CVAP'].sum()
latino_total_ar = ar_cvap['Latino CVAP'].sum()
other_total_ar = ar_cvap['Other CVAP'].sum()
print(f"black:{black_total_ar} latino:{latino_total_ar} other: {other_total_ar}")
print(f"black:{(black_total_ar > 200000)} latino:{(latino_total_ar > 200000)} other: {(other_total_ar > 200000)}")

black:330230 latino:110187 other: 96336
black:True latino:False other: False


In [5]:
ar_cvap_dict = ar_cvap.set_index("District").to_dict(orient="index")

In [46]:
ar_proportion = calc_cvap_proportion(ar_cvap_dict, state='AR')

In [47]:
ar_proportion

{'Black': {1: 0.15943725340292972,
  2: 0.20171494411151,
  3: 0.026732543862394223,
  4: 0.1951647798652985},
 'Latino': {1: 0.024432552131389555,
  2: 0.03446925254852919,
  3: 0.09752721208026759,
  4: 0.041959107589318145},
 'Other': {1: 0.0321725146969657,
  2: 0.036273070248775796,
  3: 0.0714628897728967,
  4: 0.032881552345227064}}

In [48]:
ar_overall = calc_cvap_state_proportion(ar_cvap_dict, state='AR')

In [49]:
for minority in MINORITIES:
    ar_proportion[minority]['state_wide'] = ar_overall[minority]

## Georgia

In [13]:
ga_cvap = pd.read_csv('output/Georgia/congressional_demographic.csv')
ga_cvap_dict = ga_cvap.set_index("District").to_dict(orient="index")

In [66]:
black_total_ga = ga_cvap['Black'].sum()
latino_total_ga = ga_cvap['Latino'].sum()
other_total_ga = ga_cvap['Other'].sum()
print(f"black:{black_total_ga} latino:{latino_total_ga} other: {other_total_ga}")
print(f"black:{(black_total_ga > 200000)} latino:{(latino_total_ga > 200000)} other: {(other_total_ga > 200000)}")

black:3278119 latino:1123457 other: 948176
black:True latino:True other: True


In [22]:
ga_proportion = calc_cvap_proportion(ga_cvap_dict, state='GA')

In [39]:
ga_overall = calc_cvap_state_proportion(ga_cvap_dict, state='GA')

In [51]:
for minority in MINORITIES:
    ga_proportion[minority]['state_wide'] = ga_overall[minority]

## Verify

In [ ]:
def verify_proportion(proportions, dict, state = 'AR', rel_tol=1e-9, abs_tol=1e-6):
    for district in dict:
        total = dict[int(district)]['Total CVAP'] if state=='AR' else dict[district]['Total Population'] 
        for minority in proportions:
            proportion = proportions[minority][district]
            min_pop = dict[int(district)][f'{minority} CVAP'] if state == 'AR' else dict[int(district)][minority]
            computed_total = min_pop/proportion
            if not math.isclose(computed_total, total, rel_tol=rel_tol, abs_tol=abs_tol):
                return False
    
    return True

In [35]:
print(f"verify for ga: {verify_proportion(ga_proportion, ga_cvap_dict,state='GA')}")
print(f"verify for ar: {verify_proportion(ar_proportion, ar_cvap_dict,state='AR')}")

verify for ga: True
verify for ar: True


## Export

In [57]:
ar_proportion_df = pd.DataFrame.from_dict(ar_proportion, orient='columns')
ga_proportion_df = pd.DataFrame.from_dict(ga_proportion, orient='columns')

In [60]:
ga_proportion_df

,Black,Latino,Other
1,0.275369,0.077539,0.071200
2,0.490270,0.059465,0.050844
3,0.226109,0.063106,0.067116
4,0.475444,0.192471,0.134992
5,0.497866,0.098852,0.093072
6,0.501832,0.123361,0.077006
7,0.077474,0.102365,0.182991
8,0.297241,0.071687,0.051929
9,0.119114,0.161378,0.110452
10,0.233152,0.076062,0.065984


In [59]:
ar_proportion_df.to_json('output/Arkansas/ar_cvap_proportion.json')
ga_proportion_df.to_json('output/Georgia/ga_cvap_proportion.json')